<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/06-dictionaries_pandas/06-dictionaries_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: Dictionaries and pandas

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0), and
- [pandas documentation](https://pandas.pydata.org/docs/)

```
ASK  ->  [ GET ]  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
```

So far we have used Python (Module 3), NumPy (Module 4), and seaborn (Module 5).
NumPy gave us fast arrays of numbers, but real datasets are **tables**: many rows,
several columns, and a mix of text and numbers in each row. This module gives us the
two tools that carry us through the **GET** stage and into **EXPLORE**: the Python
*dictionary* (a labeled lookup) and the pandas *DataFrame* (a spreadsheet in code).

## Learning objectives

1. Store labeled data in a **dictionary** and look values up by key.
2. Add, modify, delete, and **iterate** over key-value pairs.
3. Recognize a *record* as a dictionary and a *table* as a collection of records.
4. Build a pandas **DataFrame** from dictionaries and **read one from a CSV file**.
5. **Inspect, select, filter, and sort** a DataFrame.
6. **Create new columns** and compute **descriptive statistics**.

Our running dataset for this module is the **most-streamed songs on Spotify**. It is small
enough to read at a glance and familiar enough to reason about.

---
## 1. Dictionaries
---

### 1.1 The lookup problem

Suppose the campus **radio station** is tracking how many times some popular songs have been
streamed (in billions). A first instinct, using what we know from Module 3, is two parallel lists:

`titles  = ['Blinding Lights', 'Shape of You', 'Riptide']`

`streams = [5.3, 4.8, 3.5]`

Now answer a simple question: **how many streams does *Riptide* have?** With lists,
we first have to find *where* the title lives, then read the matching position in the other list.

In [ ]:
# RUN-TOGETHER
# Two parallel lists: the i-th title lines up with the i-th streams number.
titles  = ['Blinding Lights', 'Shape of You', 'Riptide']
streams = [5.3, 4.8, 3.5]

# To look up Riptide we must find its position first, then index the OTHER list.
i = titles.index('Riptide')     # position of the title
print('position:', i)
print('streams (billions):', streams[i])

This works, but it is fragile:

- The two lists must stay in the **same order** forever. If we sort one and forget the other,
  every lookup silently returns the wrong song.
- Looking something up takes **two steps** (find the position, then index).

We want to ask for a value *by its name*, in one step. That is exactly what a **dictionary** does.

### 1.2 Dictionaries: lookup by key

**Definition.** A *dictionary* is an unordered collection of **key-value pairs**, written in
curly braces `{ }`. Each **key** points to a **value**, and we retrieve a value by writing the
key in square brackets: `d[key]`.

- Keys must be **unique** and **immutable** (strings, ints, floats, booleans, tuples). Text keys are the most common.
- Values can be anything: numbers, strings, lists, or even other dictionaries.

Instead of two lists that must stay aligned, we bind each title to its streams figure directly.

In [ ]:
# RUN-TOGETHER
# Curly braces, and 'key : value' pairs separated by commas.
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Riptide': 3.5}

# One-step lookup by name -- no positions, no second list.
print(song_streams['Riptide'])

# The whole dictionary:
print(song_streams)

> **Why this matters.** The dictionary *is* the lookup table. There is no way for the
> keys and values to drift out of order, because each value is stored *with* its key.

#### Your Turn 1

The station wants a quick "how far ahead is the leader?" stat.

a. Using **two lookups and subtraction**, print how many billion streams `'Blinding Lights'`
   is ahead of `'Riptide'`.

b. **Predict first, then write your guess as a comment:** what does Python do if you ask for a
   title that is *not* a key, such as `song_streams['Levitating']`? (We fix this safely in the next section.)

In [ ]:
# YOUR TURN 1
# a. How far ahead is Blinding Lights over Riptide? (two lookups + subtraction)
gap = song_streams[____] - song_streams[____]
print('Blinding Lights leads Riptide by', round(gap, 1), 'billion streams')

# b. PREDICT (comment only, do not run a missing-key lookup yet):
#    song_streams['Levitating'] would do what?
#    my guess: ____

In [ ]:
# SOLUTION 1
gap = song_streams['Blinding Lights'] - song_streams['Riptide']
print('Blinding Lights leads Riptide by', round(gap, 1), 'billion streams')

# b. Asking for a missing key raises a KeyError and stops the program.
#    Section 1.3 shows the safe fix: check membership with the `in` operator first.

### 1.3 Adding, modifying, and deleting pairs

A dictionary is **changeable**. We use the same square-bracket syntax to *store* a value:

- **Add** a new pair: `d[new_key] = value`
- **Modify** an existing value: `d[existing_key] = new_value` (same syntax, an existing key)
- **Check membership**: `key in d` returns `True`/`False`
- **Delete** a pair: `del d[key]`

Why care? Real data arrives incrementally and gets corrected. Adding, fixing, and removing
entries is the day-to-day reality of maintaining a dataset.

In [ ]:
# FILL-IN-LIVE
# together: run each block and read the printout before moving on.

# ADD a song the station just started tracking.
song_streams['Die With a Smile'] = 3.5
print('after adding Die With a Smile:', song_streams)

# MEMBERSHIP test: is a key present?
print("'Die With a Smile' in song_streams? ->", 'Die With a Smile' in song_streams)
print("'Levitating' in song_streams?       ->", 'Levitating' in song_streams)

# MODIFY: streams keep climbing, so bump Die With a Smile up a little.
song_streams['Die With a Smile'] = 3.6
print('after updating Die With a Smile:', song_streams['Die With a Smile'])

# DELETE: the station stops tracking Shape of You.
del song_streams['Shape of You']
print('after deleting Shape of You:', song_streams)

#### Your Turn 2

Real datasets get *corrected*, not just overwritten. Working from the current `song_streams`:

a. **Correction without hard-coding a number.** The station realizes `'Blinding Lights'` was
   over-counted by `0.4`. Lower it using its **own current value** (read the value, subtract, store it back).

b. **Add** `'Heat Waves'` at `3.7`.

c. **Safe lookup.** Write an `if`/`else` that prints a song's streams *only if* the title is present,
   and otherwise prints "not tracked". Test it on `'Yellow'` (which is absent).

In [ ]:
# YOUR TURN 2
# a. read-modify-write: lower Blinding Lights by 0.4 using its CURRENT value (no typed number)
song_streams['Blinding Lights'] = song_streams[____] ____ 0.4

# b. add Heat Waves at 3.7
song_streams[____] = ____

# c. safe lookup guarded by `in`
title = 'Yellow'
if title ____ song_streams:
    print(title, '->', song_streams[title])
else:
    print(title, 'is not tracked')

print(song_streams)

In [ ]:
# SOLUTION 2
song_streams['Blinding Lights'] = song_streams['Blinding Lights'] - 0.4
song_streams['Heat Waves'] = 3.7

title = 'Yellow'
if title in song_streams:
    print(title, '->', song_streams[title])
else:
    print(title, 'is not tracked')

print(song_streams)

### 1.4 Iterating over a dictionary

Often we want to *walk through every pair* -- to print a report, add things up, or transform
the data. Three views help:

- `d.keys()`   -> the keys
- `d.values()` -> the values
- `d.items()`  -> the (key, value) pairs

A `for` loop over `d.items()` hands us the key and value together on each pass.

In [ ]:
# RUN-TOGETHER
# Rebuild a clean dictionary so the printout is predictable.
song_streams = {'Blinding Lights': 5.3,
                'Shape of You': 4.8,
                'Heat Waves': 3.7,
                'Riptide': 3.5,
                'Die With a Smile': 3.5}

# Loop over key/value pairs with .items()
for title, plays in song_streams.items():
    print(title, 'has about', plays, 'billion streams')

Because we can visit every value, we can also **summarize** the dictionary. Iteration is
how a lookup table becomes an answer.

In [ ]:
# FILL-IN-LIVE
# together: total the streams, and find the most-streamed song by walking the pairs.

total = 0
top_title = None
top_plays = 0

for title, plays in song_streams.items():
    total = total + plays               # running sum
    if plays > top_plays:               # track the largest so far
        top_plays = plays
        top_title = title

print('total streams (billions):', round(total, 1))
print('most-streamed:', top_title, '(', top_plays, 'billion )')

#### Your Turn 3

The demo summed the streams and tracked the top song. Now ask two *different* questions of the
same `song_streams` dictionary:

a. Compute the **average** streams. (Sum the values, then divide by the number of songs. `len(d)`
   counts the pairs.)

b. Build a **list** `big` of the titles with **more than 3.6 billion** streams, by looping over the
   pairs and appending the ones that qualify.

In [ ]:
# YOUR TURN 3
# a. average = total / number of songs
total = 0
for plays in song_streams.____():      # which view gives the values?
    total = total + plays
average = total / ____(song_streams)   # len() counts the pairs
print('average streams:', round(average, 2))

# b. collect titles above 3.6 billion into a list
big = []
for title, plays in song_streams.items():
    if plays ____ 3.6:
        big.____(title)                # add a matching title to the list
print('above 3.6 billion:', big)

In [ ]:
# SOLUTION 3
total = 0
for plays in song_streams.values():
    total = total + plays
average = total / len(song_streams)
print('average streams:', round(average, 2))

big = []
for title, plays in song_streams.items():
    if plays > 3.6:
        big.append(title)
print('above 3.6 billion:', big)

### 1.5 A record is a dictionary

So far each value has been a single number. But a dictionary can describe **one whole thing**
by using *different fields as keys*. This is called a **record**.

**Definition.** A *record* is a dictionary whose keys are **field names** (like `title`, `year`)
and whose values are that one item's data. A record is the natural stand-in for **one row of a table**.

In [ ]:
# RUN-TOGETHER
# One song described as a record: several fields, mixed value types.
blinding_lights = {'title': 'Blinding Lights',
                   'artist': 'The Weeknd',
                   'year': 2019,
                   'genre': 'Synth-pop',
                   'streams_billions': 5.3}

# Read individual fields by key.
print(blinding_lights['title'], 'was released in', blinding_lights['year'])
print('genre:', blinding_lights['genre'])

If one record is a row, then **a list of records is a table**. Here are three songs, each a record:

In [ ]:
# RUN-TOGETHER
# A list of dictionaries -- each dictionary is one row.
song_records = [
    {'title': 'Blinding Lights', 'artist': 'The Weeknd', 'year': 2019, 'streams_billions': 5.3},
    {'title': 'Riptide',         'artist': 'Vance Joy',  'year': 2013, 'streams_billions': 3.5},
    {'title': 'Bohemian Rhapsody','artist': 'Queen',     'year': 1975, 'streams_billions': 3.1},
]

# Reach into the second record (index 1), then the 'artist' field.
print(song_records[1]['title'], 'is by', song_records[1]['artist'])

#### Your Turn 4

Here is a short **list of records**. Answer two questions by **looping over the records** (no pandas
yet -- that is exactly the tedium pandas will remove in Section 2):

a. Print the `title` of every song released in **2015 or later**.

b. Find and print the `title` of the song with the **most streams** in the list.

In [ ]:
# YOUR TURN 4
catalog = [
    {'title': 'Levitating', 'artist': 'Dua Lipa',    'year': 2020, 'streams_billions': 2.6},
    {'title': 'Yellow',     'artist': 'Coldplay',    'year': 2000, 'streams_billions': 3.6},
    {'title': 'Flowers',    'artist': 'Miley Cyrus', 'year': 2023, 'streams_billions': 2.8},
    {'title': 'Creep',      'artist': 'Radiohead',   'year': 1992, 'streams_billions': 2.7},
]

# a. titles released in 2015 or later
for song in catalog:
    if song[____] >= 2015:
        print(song[____])

# b. the most-streamed song: start by assuming the first is best, then check the rest
best = catalog[0]
for song in catalog:
    if song['streams_billions'] > best[____]:
        best = song
print('most-streamed:', best['title'])

In [ ]:
# SOLUTION 4
catalog = [
    {'title': 'Levitating', 'artist': 'Dua Lipa',    'year': 2020, 'streams_billions': 2.6},
    {'title': 'Yellow',     'artist': 'Coldplay',    'year': 2000, 'streams_billions': 3.6},
    {'title': 'Flowers',    'artist': 'Miley Cyrus', 'year': 2023, 'streams_billions': 2.8},
    {'title': 'Creep',      'artist': 'Radiohead',   'year': 1992, 'streams_billions': 2.7},
]

for song in catalog:
    if song['year'] >= 2015:
        print(song['title'])

best = catalog[0]
for song in catalog:
    if song['streams_billions'] > best['streams_billions']:
        best = song
print('most-streamed:', best['title'])

### Lists vs. Dictionaries: which one?

| | Indexed by | Order matters? | Good for |
| --- | --- | --- | --- |
| **List** | position (`0, 1, 2, ...`) | yes | a sequence of values (a column) |
| **Dictionary** | unique key | no | a fast lookup table; a record with named fields |

Rule of thumb: if you find yourself keeping two lists *in sync* so that position `i` in one
matches position `i` in another, you probably want a dictionary (or, very soon, a DataFrame).

---
## 2. From Dictionaries to pandas DataFrames
---

### 2.1 Why pandas?

A list of records is a table *in spirit*, but doing table work with raw dictionaries
(filtering rows, averaging a column) quickly gets clumsy. **pandas** is the standard Python
library for tabular data.

**Definition.** A pandas *DataFrame* is a two-dimensional table of data.
- Each **row** is one observation (here, one song).
- Each **column** is one feature (title, artist, year, ...), and has a label.
- It is built on top of NumPy, so column math is fast and element-wise, just like Module 4.

By convention we import it as `pd`.

In [ ]:
# RUN-TOGETHER
import pandas as pd

### 2.2 Building a DataFrame from a dictionary

The most common pattern is a **dictionary of columns**: each **key is a column name**, and each
**value is a list** holding that column's data, top to bottom.

In [ ]:
# RUN-TOGETHER
# A dictionary of columns: key = column name, value = list of column data.
data = {
    'title':  ['Blinding Lights', 'Shape of You', 'Riptide', 'Die With a Smile', 'Bohemian Rhapsody'],
    'artist': ['The Weeknd', 'Ed Sheeran', 'Vance Joy', 'Lady Gaga & Bruno Mars', 'Queen'],
    'year':   [2019, 2017, 2013, 2024, 1975],
    'streams_billions': [5.3, 4.8, 3.5, 3.5, 3.1],
}

songs_small = pd.DataFrame(data)
songs_small   # a bare DataFrame on the last line renders as a nice table

There are two other layouts worth knowing:

1. **A list of records** (list of dictionaries) -- pandas lines the fields up into columns for you.
2. **A dictionary of records** (dictionary of dictionaries) -- here the *keys become the columns*,
   so we usually **transpose** with `.T` to get one song per row. This one is a common trap, so we
   show the clean layouts first.

In [ ]:
# RUN-TOGETHER
# Layout 1: list of records -> DataFrame (fields become columns automatically).
records = [
    {'title': 'Blinding Lights', 'year': 2019, 'streams_billions': 5.3},
    {'title': 'Riptide',         'year': 2013, 'streams_billions': 3.5},
]
print(pd.DataFrame(records))

# Layout 2: dictionary of records -> the KEYS become columns, so we transpose with .T.
by_title = {
    'Blinding Lights': {'year': 2019, 'streams_billions': 5.3},
    'Riptide':         {'year': 2013, 'streams_billions': 3.5},
}
print()
print(pd.DataFrame(by_title).T)   # .T flips rows and columns so each song is a row

#### Your Turn 5 (spot the bug)

Every column in a dictionary-of-columns must be the **same length**. The dictionary below breaks
that rule -- `year` has only two values while the others have three. If you built it as written,
pandas would stop with:

```
ValueError: All arrays must be of the same length
```

Read that message, then **fix it** by supplying the missing `year` for *Parasite* (it is **2019**).

In [ ]:
# YOUR TURN 5 -- fix the length mismatch
playlist = {
    'title':  ['Spirited Away', 'The Matrix', 'Parasite'],
    'year':   [2001, 1999, ____],     # <- one value is missing; add it
    'runtime_min': [125, 136, 132],
}
pd.DataFrame(playlist)

In [ ]:
# SOLUTION 5
playlist = {
    'title':  ['Spirited Away', 'The Matrix', 'Parasite'],
    'year':   [2001, 1999, 2019],
    'runtime_min': [125, 136, 132],
}
pd.DataFrame(playlist)

### 2.3 Meeting a full dataset, and inspecting it

Let's load the station's full list of most-streamed songs. We build it once as a dictionary of
columns, then turn it into a DataFrame called `songs`.

In [ ]:
# RUN-TOGETHER
# The full dataset (a dictionary of columns).
songs_data = {
    'title': ['Blinding Lights', 'Shape of You', 'Perfect', 'Believer', 'Heat Waves',
              'Yellow', 'Riptide', 'Die With a Smile', 'Take Me to Church', 'Die For You',
              'Bohemian Rhapsody', 'Mr. Brightside'],
    'artist': ['The Weeknd', 'Ed Sheeran', 'Ed Sheeran', 'Imagine Dragons', 'Glass Animals',
               'Coldplay', 'Vance Joy', 'Lady Gaga & Bruno Mars', 'Hozier', 'The Weeknd',
               'Queen', 'The Killers'],
    'year': [2019, 2017, 2017, 2017, 2020, 2000, 2013, 2024, 2013, 2016, 1975, 2004],
    'genre': ['Synth-pop', 'Pop', 'Pop', 'Pop rock', 'Indie pop', 'Alternative rock',
              'Indie folk', 'Pop', 'Indie rock', 'R&B', 'Rock', 'Rock'],
    'streams_billions': [5.3, 4.8, 3.9, 3.8, 3.7, 3.6, 3.5, 3.5, 3.4, 3.2, 3.1, 3.0],
}

songs = pd.DataFrame(songs_data)
songs

> **Note on the data.** Titles, artists, and release years are factual. The
> `streams_billions` figures are **approximate, rounded** cumulative Spotify play counts from a
> single snapshot in early 2026 (about March 2026). Streaming totals climb every day and differ
> by source, so treat them as ballpark figures for practice, not precise citations. **Genre** is
> a single simplified label; most of these songs blend several styles, so the labels are broad
> strokes, not official classifications.

When you meet a table, look before you leap. These are the first commands to run on any new DataFrame:

- `df.head(n)`  -> the first `n` rows (default 5)
- `df.shape`    -> `(rows, columns)` -- an **attribute**, so no parentheses (recall Module 4)
- `df.columns`  -> the column labels
- `df.dtypes`   -> the type stored in each column
- `df.info()`   -> a compact summary: columns, non-null counts, dtypes
- `df.describe()` -> descriptive statistics for the numeric columns

In [ ]:
# RUN-TOGETHER
print('shape (rows, cols):', songs.shape)     # attribute: no ()
print()
print('column labels:', list(songs.columns))
print()
print('dtypes:')
print(songs.dtypes)

In [ ]:
# RUN-TOGETHER
songs.head(3)   # first three rows

In [ ]:
# RUN-TOGETHER
songs.info()    # structure + non-null counts (great for spotting missing data)

#### Your Turn 6

Inspection is only useful if you can **read numbers back out** of it. `songs.describe()` returns a
table you can index with `.loc[row_label, column]`, where the row labels are `'mean'`, `'50%'`
(the median), `'max'`, and so on.

a. Pull out and print the **mean release year** and the **maximum streams** value.

b. Print the **mean** and the **median** (`'50%'`) of `streams_billions`. Then, in a comment, say
   which is larger and what that implies about the **shape** of the streams distribution.

In [ ]:
# YOUR TURN 6
d = songs.describe()

# a. read single numbers out of the describe() table with .loc
print('mean release year:', round(d.loc['mean', 'year'], 1))
print('max streams:', d.loc[____, 'streams_billions'])

# b. mean vs median of streams
print('mean streams:  ', round(d.loc['mean', 'streams_billions'], 2))
print('median streams:', d.loc[____, 'streams_billions'])
# Which is larger, and what does that say about the distribution?
# your answer: ____

In [ ]:
# SOLUTION 6
d = songs.describe()
print('mean release year:', round(d.loc['mean', 'year'], 1))
print('max streams:', d.loc['max', 'streams_billions'])
print('mean streams:  ', round(d.loc['mean', 'streams_billions'], 2))
print('median streams:', d.loc['50%', 'streams_billions'])
# The mean is larger than the median: a few very high values (Blinding Lights, Shape of You)
# pull the mean up, so the streams distribution is right-skewed.

### 2.4 Reading data from a CSV file

Typing a dataset by hand is fine for a demo, but in practice data lives in **files**. The most
common is a **CSV** (comma-separated values): a plain-text table where columns are separated by
commas and each line is a row.

We rarely build the CSV ourselves, but doing the round trip once shows exactly what
`read_csv` reads back. We **save** `songs` to a file, then **read** it into a fresh DataFrame.

In [ ]:
# RUN-TOGETHER
# Save our DataFrame to a CSV file. index=False keeps pandas from writing the row numbers.
songs.to_csv('songs.csv', index=False)

# Read it back. In real projects you START here, with a file someone handed you.
songs = pd.read_csv('songs.csv')

# ALWAYS check that it loaded the way you expected.
songs.head()

A few practical notes on `read_csv`:

- **Reading from the web.** If a file lives online, pass the URL directly. For example, once the
  station commits `songs.csv` to the course repository, students can load it with:

  ```python
  url = 'https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/06-dictionaries_pandas/songs.csv'
  songs = pd.read_csv(url)
  ```
  (No Google Drive mounting required.)

- **The index gotcha.** If a CSV already contains a leftmost column of row labels, tell pandas
  with `pd.read_csv(path, index_col=0)` so those labels become the row index instead of a data column.

- **Always inspect after loading** (`head`, `shape`, `dtypes`). Silent misreads are the most
  common early bug.

---
## 3. Exploring a DataFrame
---

Now the payoff. With one clean table we can select, filter, sort, derive, and summarize --
the core moves of the EXPLORE stage.

### 3.1 Selecting columns

- **Single brackets** with one column name -> a **Series** (a labeled 1-D column).
- **Double brackets** with a list of names -> a **DataFrame** (a table, even for one column).

Think of it as: *"one thing in the brackets, get a column; a list in the brackets, get a table."*

In [ ]:
# RUN-TOGETHER
# Single brackets -> Series
titles = songs['title']
print(type(titles))     # pandas Series (a labeled 1-D array)
print(titles.head(3))

print()
# Double brackets -> DataFrame (note the list inside)
subset = songs[['title', 'streams_billions']]
print(type(subset))     # pandas DataFrame
subset.head(3)

#### Your Turn 7 (predict, then run)

The single-vs-double bracket rule is the most common early confusion, so *commit to a guess first*.
For each expression below, **predict** whether it returns a **Series** or a **DataFrame** and write
`S` or `D` in the comment. Then run the cell and check yourself against `type(...)`.

In [ ]:
# YOUR TURN 7 -- predict S (Series) or D (DataFrame) for each, then run
print(type(songs['artist']))              # your guess: ____
print(type(songs[['artist']]))            # your guess: ____
print(type(songs[['title', 'artist']]))   # your guess: ____

In [ ]:
# SOLUTION 7
# One name in single brackets -> Series. A LIST inside the brackets -> DataFrame,
# even when the list holds a single column. So the answers are: Series, DataFrame, DataFrame.
print(type(songs['artist']))              # Series
print(type(songs[['artist']]))            # DataFrame
print(type(songs[['title', 'artist']]))   # DataFrame

### 3.2 Selecting rows: `.iloc` and `.loc`

Two row selectors, mirroring the difference between *position* and *label*:

- `df.iloc[i]`  -> row by **integer position** (like list indexing; 0-based).
- `df.loc[label]` -> row by **index label**.

By default a DataFrame's index is `0, 1, 2, ...`, so `loc` and `iloc` look alike. They diverge
once we set a meaningful index -- here, the song's title.

In [ ]:
# RUN-TOGETHER
# Position-based: the first row, and rows 0..2.
print('first row (iloc[0]):')
print(songs.iloc[0])
print()
print('first three rows (iloc[0:3]):')
print(songs.iloc[0:3][['title', 'streams_billions']])

In [ ]:
# RUN-TOGETHER
# Make the title the row label, then select by that label with .loc.
songs_by_title = songs.set_index('title')
print(songs_by_title.loc['Bohemian Rhapsody'])          # one row, selected by its NAME
print()
print(songs_by_title.loc[['Blinding Lights', 'Riptide'], ['year', 'streams_billions']])

#### Your Turn 8

The point of `.iloc` vs `.loc` is **position vs label**. Using `songs_by_title` (title is the index):

a. Show that **position 0** and the **label `'Blinding Lights'`** point to the *same* row by printing
   its streams both ways.

b. Only one of `songs_by_title.loc[0]` and `songs_by_title.iloc[0]` works here. Keep the one that
   **works**, and write a one-line comment saying why the other one fails.

In [ ]:
# YOUR TURN 8
songs_by_title = songs.set_index('title')

# a. same row, two ways: position 0 and the label sitting at position 0
print(songs_by_title.iloc[0]['streams_billions'])
print(songs_by_title.loc[____]['streams_billions'])   # use the LABEL

# b. keep the selector that works (loc wants a label; our labels are titles, not numbers)
print(songs_by_title.____[0]['year'])
# why does the other one fail? your reason: ____

In [ ]:
# SOLUTION 8
songs_by_title = songs.set_index('title')

print(songs_by_title.iloc[0]['streams_billions'])
print(songs_by_title.loc['Blinding Lights']['streams_billions'])

# iloc uses POSITION, so iloc[0] works. loc[0] would fail: after set_index the labels are
# titles, so 0 is not a valid label.
print(songs_by_title.iloc[0]['year'])

### 3.3 Filtering rows with boolean masks

This is the same idea as NumPy boolean masks from Module 4, now on a whole table. A comparison
on a column produces a column of `True`/`False`; putting that mask in the brackets keeps only the
`True` rows.

- Combine conditions with `&` (and) and `|` (or), and **wrap each condition in parentheses**.

In [ ]:
# RUN-TOGETHER
# Step 1: a comparison makes a boolean column.
mask = songs['streams_billions'] > 4
print(mask.head())

print()
# Step 2: use the mask to keep matching rows.
mega = songs[songs['streams_billions'] > 4]
print('songs over 4 billion streams:', mega.shape[0])
mega[['title', 'streams_billions']]

In [ ]:
# RUN-TOGETHER
# Filter on text, and combine two conditions with & (parentheses required!).
pop = songs[songs['genre'] == 'Pop']
print('Pop titles:', list(pop['title']))

recent_pop = songs[(songs['genre'] == 'Pop') & (songs['year'] >= 2020)]
recent_pop[['title', 'year']]

#### Your Turn 9

The demo used `&` (and). Now use `|` (or), and **predict before you run**.

a. Find songs that are **either** released **before 2005** *or* have **more than 4.5 billion**
   streams. Before running, guess how many rows you will get.

b. In a comment, explain why each condition must be wrapped in its **own parentheses**. (Hint:
   what would `songs['year'] < 2005 | songs['streams_billions'] > 4.5` try to do first?)

In [ ]:
# YOUR TURN 9
# a. before 2005 OR more than 4.5 billion -- predict the count first!
picks = songs[(songs['year'] < 2005) ____ (songs['streams_billions'] > 4.5)]
print('count:', picks.shape[0])
picks[['title', 'year', 'streams_billions']]

In [ ]:
# SOLUTION 9
picks = songs[(songs['year'] < 2005) | (songs['streams_billions'] > 4.5)]
print('count:', picks.shape[0])
display(picks[['title', 'year', 'streams_billions']])

# Parentheses are required because `|` binds TIGHTER than `<` and `>`. Without them, Python
# tries to evaluate 2005 | songs['streams_billions'] first, which is meaningless and errors.

### 3.4 Sorting

`df.sort_values('column')` returns a new DataFrame ordered by that column. Add `ascending=False`
for largest-first. Sorting answers "top" and "bottom" questions at a glance.

In [ ]:
# RUN-TOGETHER
# Most-streamed first.
top = songs.sort_values('streams_billions', ascending=False)
top[['title', 'streams_billions']].head()

In [ ]:
# RUN-TOGETHER
# Oldest songs first (ascending is the default).
songs.sort_values('year')[['title', 'year']].head()

#### Your Turn 10

a. **Sort by two keys at once.** Order by `genre` (A-Z), and *within* each genre by
   `streams_billions` (highest first). Pass a **list of columns** and a **list of directions**
   to `ascending` (e.g. `ascending=[True, False]`).

b. Using **sort + position**, print the `title` of the **3rd-most-streamed** song overall.

In [ ]:
# YOUR TURN 10
# a. genre A-Z, then streams high-to-low within each genre
by_genre = songs.sort_values(['genre', 'streams_billions'], ascending=[____, ____])
display(by_genre[['genre', 'title', 'streams_billions']])

# b. 3rd-most-streamed overall: sort high-to-low, then take position 2 (0, 1, 2, ...)
third = songs.sort_values('streams_billions', ascending=False).iloc[____]
print('3rd most-streamed:', third['title'])

In [ ]:
# SOLUTION 10
by_genre = songs.sort_values(['genre', 'streams_billions'], ascending=[True, False])
display(by_genre[['genre', 'title', 'streams_billions']])

third = songs.sort_values('streams_billions', ascending=False).iloc[2]
print('3rd most-streamed:', third['title'])

### 3.5 Creating new columns

We often need a value the raw data does not contain, computed from columns we do have. Assigning
to a *new* column name creates it. Column math is element-wise (thanks to NumPy underneath), so
no loop is needed.

Why bother? A derived column can reframe the question. Total streams favor older songs that have
had years to accumulate plays; **streams per year** is a fairer "how fast is it being played"
measure. Building useful new columns like this is the beginning of *feature engineering*, which
we return to later in the course.

In [ ]:
# RUN-TOGETHER
# It is 2026. Age of each song, in years.
songs['age_years'] = 2026 - songs['year']

# A rate: average billions of streams PER year on the platform.
songs['streams_per_year'] = (songs['streams_billions'] / songs['age_years']).round(2)

# A boolean flag column.
songs['is_recent'] = songs['year'] >= 2015

songs[['title', 'year', 'age_years', 'streams_billions', 'streams_per_year', 'is_recent']].head()

> **Discuss.** Sort by `streams_billions` and then by `streams_per_year`. Does the same song
> sit on top both times? What does each ranking reward?

#### Your Turn 11

Build a column to *answer a question*, then interpret a rate.

a. Create `gap_to_leader` = how far each song trails the top song (the **max** streams minus that
   song's streams). Then filter to the songs **within 1.0 billion** of the leader.

b. Sort by `streams_per_year` (highest first). In a comment, name the song on top and explain how a
   **2024** release can out-rank songs with far more *total* streams.

In [ ]:
# YOUR TURN 11
# a. distance behind the most-streamed song
leader = songs['streams_billions'].____()          # the max value
songs['gap_to_leader'] = (leader - songs['streams_billions']).round(1)
close = songs[songs['gap_to_leader'] ____ 1.0]     # within 1 billion of the top
display(close[['title', 'streams_billions', 'gap_to_leader']])

# b. fastest accumulators: streams per year, highest first
songs['streams_per_year'] = (songs['streams_billions'] / (2026 - songs['year'])).round(2)
display(songs.sort_values('streams_per_year', ascending=False)[['title', 'year', 'streams_per_year']].head(3))
# Which song leads per year, and why can a 2024 song beat older giants?
# your answer: ____

In [ ]:
# SOLUTION 11
leader = songs['streams_billions'].max()
songs['gap_to_leader'] = (leader - songs['streams_billions']).round(1)
close = songs[songs['gap_to_leader'] <= 1.0]
display(close[['title', 'streams_billions', 'gap_to_leader']])

songs['streams_per_year'] = (songs['streams_billions'] / (2026 - songs['year'])).round(2)
display(songs.sort_values('streams_per_year', ascending=False)[['title', 'year', 'streams_per_year']].head(3))
# Die With a Smile (2024) leads per year: it gathered ~3.5B in only about two years, so its
# per-year rate is high even though its TOTAL still trails Blinding Lights.

### 3.6 Summaries and descriptive statistics

A few workhorses:

- `df['col'].mean()`, `.median()`, `.min()`, `.max()`, `.sum()` -> one number about a column.
- `df['col'].value_counts()` -> counts of each category in a text column.
- `df.groupby('group_col')['value_col'].mean()` -> one summary *per group*.
- `df.describe()` -> count, mean, std, min, quartiles, max for every numeric column at once.

In [ ]:
# RUN-TOGETHER
print('mean streams (billions):  ', round(songs['streams_billions'].mean(), 2))
print('median streams (billions):', songs['streams_billions'].median())
print('newest release year:', songs['year'].max())
print()

# How many songs per genre? (a category count)
print('songs per genre:')
print(songs['genre'].value_counts())

In [ ]:
# RUN-TOGETHER
# groupby: average streams per genre (one number per genre).
avg_by_genre = songs.groupby('genre')['streams_billions'].mean().round(2)
print(avg_by_genre.sort_values(ascending=False))

#### Your Turn 12

Use grouping to reach a *conclusion*, not just a table. `idxmax()` returns the **label** of the
largest value (handy for "which group wins").

a. Which **genre** has the highest **average** streams? Get its name directly with `.idxmax()`.

b. Which **artist** appears **most often**? (`value_counts()` then `.idxmax()`.)

c. Print the average streams **per artist**. In a comment, answer: is the most *frequent* artist also
   the one with the highest *average*? (They are different questions -- watch what the numbers say.)

In [ ]:
# YOUR TURN 12
# a. genre with the highest average streams (the name, not the whole table)
avg_by_genre = songs.groupby('genre')['streams_billions'].mean()
print('top genre by avg streams:', avg_by_genre.____())     # idxmax -> the winning label

# b. most frequent artist
print('most frequent artist:', songs['artist'].value_counts().____())

# c. average streams per artist, highest first
print(songs.groupby(____)['streams_billions'].mean().round(2).sort_values(ascending=False))
# Is 'most frequent' the same as 'highest average'? your conclusion: ____

In [ ]:
# SOLUTION 12
avg_by_genre = songs.groupby('genre')['streams_billions'].mean()
print('top genre by avg streams:', avg_by_genre.idxmax())

print('most frequent artist:', songs['artist'].value_counts().idxmax())

print(songs.groupby('artist')['streams_billions'].mean().round(2).sort_values(ascending=False))
# Two subtleties worth noticing:
#  - idxmax picks 'Synth-pop', but that genre holds only ONE song, so its "average" is a single
#    value. Averages over size-1 groups are unstable -- always check the group size.
#  - The Weeknd and Ed Sheeran tie as most frequent (2 songs each); Ed Sheeran also has the
#    highest per-artist average, The Weeknd does not. "Most frequent" and "highest average"
#    are different questions that need not agree.

---
## 4. Guided Practice: A Realistic Workflow
---

### The scenario

You volunteer at the campus **radio station**, which is building a "**Featured Tracks**"
playlist for its homepage. A staffer collected a fresh batch of songs and handed you the
notes as a **list of records** (a list of dictionaries) -- exactly the messy-but-common format
real data arrives in.

Your job walks through a realistic mini-pipeline:

- **Part A.** Turn the raw records into a DataFrame and inspect it. *(dictionaries -> DataFrame)*
- **Part B.** Enrich it with a derived column and a **lookup dictionary**, then filter and sort.
- **Part C.** Summarize and write a one-sentence recommendation.

**Predict each output before you run it.**

> The `station_score` values below are the station's own made-up ratings (out of 10), included so
> we have a numeric column to sort and average. Titles, artists, years, and genres are factual.

#### Part A. From raw records to a table

Build a DataFrame from the list of records and get oriented.

In [ ]:
# Part A -- fill in the blanks
import pandas as pd

# Raw notes from a station staffer: a list of dictionaries (one per song).
raw = [
    {'title': 'As It Was',   'artist': 'Harry Styles',              'year': 2022, 'genre': 'Pop'},
    {'title': 'Levitating',  'artist': 'Dua Lipa',                  'year': 2020, 'genre': 'Disco-pop'},
    {'title': 'good 4 u',    'artist': 'Olivia Rodrigo',            'year': 2021, 'genre': 'Pop rock'},
    {'title': 'Sunflower',   'artist': 'Post Malone & Swae Lee',    'year': 2018, 'genre': 'Hip-hop'},
    {'title': 'Uptown Funk', 'artist': 'Mark Ronson ft. Bruno Mars','year': 2014, 'genre': 'Funk'},
    {'title': 'Kill Bill',   'artist': 'SZA',                       'year': 2022, 'genre': 'R&B'},
]

# 1. Build a DataFrame from the list of records.
station = pd.DataFrame(____)

# 2. Inspect: shape, then the first few rows.
print('shape:', station.____)
station.head()

In [ ]:
# Part A -- SOLUTION
import pandas as pd

raw = [
    {'title': 'As It Was',   'artist': 'Harry Styles',              'year': 2022, 'genre': 'Pop'},
    {'title': 'Levitating',  'artist': 'Dua Lipa',                  'year': 2020, 'genre': 'Disco-pop'},
    {'title': 'good 4 u',    'artist': 'Olivia Rodrigo',            'year': 2021, 'genre': 'Pop rock'},
    {'title': 'Sunflower',   'artist': 'Post Malone & Swae Lee',    'year': 2018, 'genre': 'Hip-hop'},
    {'title': 'Uptown Funk', 'artist': 'Mark Ronson ft. Bruno Mars','year': 2014, 'genre': 'Funk'},
    {'title': 'Kill Bill',   'artist': 'SZA',                       'year': 2022, 'genre': 'R&B'},
]

station = pd.DataFrame(raw)
print('shape:', station.shape)
station.head()

#### Part B. Enrich with a derived column and a lookup dictionary

Two moves that combine everything so far:

1. A **derived column** `age_years` (it is 2026).
2. A **lookup dictionary** that maps each title to the station's rating, attached with `.map`.
   `df['title'].map(some_dict)` walks the `title` column and replaces each title with its value
   from the dictionary -- a dictionary lookup applied to a whole column at once.

Then filter to recent songs and sort by the station's score.

In [ ]:
# Part B -- fill in the blanks

# 1. Derived column: how old is each song in 2026?
station['age_years'] = 2026 - station[____]

# 2. A lookup dictionary: title -> station rating (out of 10, made up for this exercise).
station_score = {
    'As It Was': 9,
    'Levitating': 8,
    'good 4 u': 9,
    'Sunflower': 10,
    'Uptown Funk': 10,
    'Kill Bill': 9,
}
# Map the dictionary onto the title column to create a new column.
station['station_score'] = station['title'].map(____)

# 3. Filter to songs from 2018 or later, then sort by station_score (highest first).
recent = station[station['year'] >= ____]
recent.sort_values(____, ascending=False)

In [ ]:
# Part B -- SOLUTION
station['age_years'] = 2026 - station['year']

station_score = {
    'As It Was': 9,
    'Levitating': 8,
    'good 4 u': 9,
    'Sunflower': 10,
    'Uptown Funk': 10,
    'Kill Bill': 9,
}
station['station_score'] = station['title'].map(station_score)

recent = station[station['year'] >= 2018]
recent.sort_values('station_score', ascending=False)

#### Part C. Summarize and recommend

Answer the station's questions, then write your recommendation as a comment.

1. What is the **average** `station_score` across all songs?
2. How many songs are there **per genre**?
3. Which single song would you feature first? Use the sorted table from Part B to justify it.

In [ ]:
# Part C -- fill in the blanks

# 1. Average station score across all songs.
print('mean station score:', round(station['station_score'].____(), 2))

# 2. Songs per genre.
print()
print(station[____].value_counts())

# 3. Top pick: highest station_score, tie broken by newest year.
top_pick = station.sort_values(['station_score', 'year'], ascending=False).____(1)
print()
print(top_pick[['title', 'year', 'genre', 'station_score']])

# Our recommendation (write one sentence as a comment):
#

In [ ]:
# Part C -- SOLUTION
print('mean station score:', round(station['station_score'].mean(), 2))
print()
print(station['genre'].value_counts())

top_pick = station.sort_values(['station_score', 'year'], ascending=False).head(1)
print()
print(top_pick[['title', 'year', 'genre', 'station_score']])

# Our recommendation (example):
# Feature Sunflower first: it ties for the station's top score (10/10) and is the
# newer of the two top-rated songs, so it is the most likely to draw current listeners.

#### Part D. Stretch challenge: find a "hidden gem"

The station wants a **hidden gem** for the playlist: a highly rated song that is *not* one of the
obvious recent hits. Find the song with the **highest `station_score` among those released before
2020**; if two tie on score, prefer the **newer** one. Print its title, year, and score, then pitch
it in one sentence.

There is more than one reasonable way to do this -- try it before peeking.

In [ ]:
# Part D -- fill in the blanks
gems = station[station['year'] < ____]      # released before 2020
top_gem = gems.sort_values(['station_score', 'year'], ascending=False).____(1)
print(top_gem[['title', 'year', 'station_score']])

# Your one-sentence pitch (as a comment):
#

In [ ]:
# Part D -- SOLUTION
gems = station[station['year'] < 2020]
top_gem = gems.sort_values(['station_score', 'year'], ascending=False).head(1)
print(top_gem[['title', 'year', 'station_score']])

# Pitch (example):
# Feature Sunflower: it matches the top score in the whole set (10/10) but predates the recent-hit
# wave, so it stands out as a familiar-yet-unexpected pick rather than another chart-topper.

---
## Recap and what's next

You can now move data through the first half of the pipeline on your own:

- **Dictionaries** give you labeled lookups and records (rows).
- A **DataFrame** collects records into a table you can read from a **CSV**.
- You can **inspect, select, filter, sort**, build **new columns**, and compute **summaries**.

Everything we plotted in Module 5 lives in a DataFrame, and every column you selected or filtered
here can be handed straight to seaborn. Next we combine these skills to **clean and explore a full,
messy dataset** end to end.